# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² mass spectrometry dataset using the `mlcroissant` library. All dataset components are referenced by their Croissant `@id` to ensure transparency and reproducibility.

### Dataset Source
* Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and preview its structure using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# View top-level metadata
print(f"Dataset Name: {dataset.metadata.name}\n")
print(f"Dataset Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview

List available record sets and the fields/columns within each, using their `@id` values for precise reference.

In [ ]:
# List all record sets in the dataset by @id and name
print("Available Record Sets:")
for rs in dataset.metadata.recordSets:
    print(f"  Record Set @id: {rs.id}, name: {getattr(rs, 'name', '(no name)')}")
    # List fields (columns) for each record set
    if hasattr(rs, 'fields'):
        print("    Fields and Columns:")
        for field in rs.fields:
            fname = getattr(field, 'name', '(no name)')
            fdesc = getattr(field, 'description', '')
            print(f"      Field @id: {field.id}, name: {fname} - {fdesc}")
        print("")

## 3. Data Extraction

Load data from the main record set into a DataFrame for analysis. We'll use the record set and fields' `@id` values as explored above.

In [ ]:
# 1. Collect all record set @id's
record_sets = [rs.id for rs in dataset.metadata.recordSets]
dataframes = {}

# 2. Extract data for each record set
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set {record_set_id}")

# 3. Choose main record set for analysis (use first listed if unsure)
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id:
    print(f"\nColumns for {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Typical data processing includes filtering records, normalizing numeric fields, removing outliers, or grouping data by a field for summary. All columns are referenced by their `@id`.

**Example:** Filter and normalize on a numeric field (e.g., `mw` - molecular weight), and group by a categorical field (e.g., `accession`).

In [ ]:
# For demonstration, extract the molecular weight field and group by accession

# Replace these with the actual field @id's for molecular weight and accession after printing the overview above
numeric_field_id = None  # e.g., '@id' for MW (molecular weight), often 'mw'
group_field_id = None    # e.g., '@id' for accession, often 'accession'

# If you printed the field @id's above, fill in the right values here:
# Example (replace with actual):
# numeric_field_id = 'https://api.app.sen.science/frontiers/7154140/field/mw'
# group_field_id = 'https://api.app.sen.science/frontiers/7154140/field/accession'

# For now, auto-detect numeric columns likely to be molecular weights
main_df = dataframes[main_record_set_id]
numeric_candidates = [col for col in main_df.columns if 'mw' in col.lower() or 'weight' in col.lower()]
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    # Otherwise, pick the first float column
    for c in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[c]):
            numeric_field_id = c
            break

if group_field_id is None:
    # Try to auto-detect accession or similar
    for col in main_df.columns:
        if 'accession' in col.lower():
            group_field_id = col
            break

print(f"Numeric field used: {numeric_field_id}")
print(f"Grouping field: {group_field_id}")

# Filter records with numeric_field > threshold (e.g., MW > 10)
threshold = 10
filtered_df = main_df
if numeric_field_id:
    filtered_df = filtered_df[pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows")

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
    ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()

    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field (e.g., protein accession)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the numeric field (e.g., molecular weight) and its relationship to other fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in main_df:
    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(main_df[numeric_field_id], errors='coerce').dropna(), bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # If group_field is categorical and not too many groups, show boxplot
    if group_field_id and group_field_id in main_df and main_df[group_field_id].nunique() <= 30:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=main_df[group_field_id], y=pd.to_numeric(main_df[numeric_field_id], errors='coerce'))
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45, ha='right')
        plt.show()

## 6. Conclusion

Using `mlcroissant`, we systematically loaded, explored, and visualized human protein mass spectrometry data with full traceability via Croissant `@id` links. This workflow enables transparent and reproducible FAIR data usage. You can now tailor the EDA and processing for deeper biological or analytical insights.